# Verificação de ausência de vazamento de dados

Este notebook reúne as verificações que garantem que nenhuma informação do conjunto de teste alcança o treino. Ele não prepara dados nem treina modelos: apenas examina os artefatos já produzidos, que são a partição gravada e os conjuntos por área.

## Por que cada verificação vem acompanhada de um caso de controle

Uma sequência de asserções que passa não prova nada por si só, pois uma verificação mal escrita passa sempre. Por essa razão, cada verificação é executada duas vezes: uma sobre os artefatos reais, em que deve passar, e outra sobre uma cópia corrompida de propósito, em que deve falhar.

O resultado esperado é, portanto, uma tabela em que a coluna dos dados reais apresenta aprovação e a coluna dos dados corrompidos apresenta reprovação. Uma verificação que passe também sobre os dados corrompidos está quebrada, e o notebook interrompe a execução nesse caso.

## Implementação independente

As verificações reimplementam as regras em vez de importá-las do notebook de divisão. A duplicação é deliberada: um teste que compartilha código com aquilo que testa deixa de detectar erros presentes nesse código comum.

## O que é verificado

| Verificação | Vazamento que ela impede |
|---|---|
| Disjunção por ensaio | o mesmo ensaio aparecer em treino e em teste |
| Disjunção por configuração, chave do perfil | o modelo ver em treino uma configuração idêntica à de teste |
| Disjunção por configuração, chave de 13 categorias | a partição ser válida para uma versão da base e inválida para a outra |
| Cobertura da partição | ensaios ficarem fora de ambos os conjuntos, ou duplicados |
| Representação dos tipos de ensaio | uma dobra ser avaliada por extrapolação pura |
| Ausência de feature derivada do alvo | o valor a prever entrar disfarçado no conjunto de entrada |
| Imputação ajustada no treino | a estatística de preenchimento incorporar valores do teste |
| Avaliação sem valor imputado | o modelo ser pontuado por reproduzir um rótulo criado pela equipe |

## 1. Configuração e leitura dos artefatos

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/preparacao_testes"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os

import numpy as np
import pandas as pd

PASTA_DADOS = "colabs_S3"
ARQ_PERFIL = "dataset_mainback_perfil.csv"
ARQ_13CAT = "dataset_mainback_13cat.csv"
ARQ_ALVOS = "dataset_ipc_comp_13cat.csv"   # area com mais colunas-alvo e mais ausencias
ALVO_DEMO = "Heat Sink (Point/Probe)"      # coluna de menor cobertura da base

PREFIXOS_CONFIG = ("blower_direction_", "housing_", "board_position_",
                   "run_case_", "losses_", "perfil_perdas_",
                   "deflector_hole_", "material_coil_support_", "cfg_")

BASES = [".", "..", os.path.join("..", ".."),
         "notebooks", os.path.join("..", "notebooks"),
         "preparacao_testes", os.path.join("..", "preparacao_testes"),
         os.path.join("notebooks", "preparacao_testes")]


def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError("Arquivo nao encontrado: " + arquivo
                            + ". Diretorio atual: " + os.path.abspath(".")
                            + ". Caminhos tentados: " + "; ".join(tentativas))


def colunas_de_configuracao(df):
    return [c for c in df.columns if c.startswith(PREFIXOS_CONFIG)]


def chave(df):
    return df[colunas_de_configuracao(df)].astype(str).agg("|".join, axis=1)


particao = pd.read_csv(resolve("preparacao_testes", "particao_treino_teste.csv"))
perfil = pd.read_csv(resolve(PASTA_DADOS, ARQ_PERFIL))
cat13 = pd.read_csv(resolve(PASTA_DADOS, ARQ_13CAT))
alvos = pd.read_csv(resolve(PASTA_DADOS, ARQ_ALVOS))

print("Particao:", particao.shape)
print(particao.groupby("conjunto").size().to_string())
print()
print("Conjuntos por area lidos para conferencia:", ARQ_PERFIL, "|", ARQ_13CAT, "|", ARQ_ALVOS)

## 2. As verificações

Cada função abaixo recebe a partição e levanta uma exceção quando a propriedade que ela protege é violada. Nenhuma delas imprime nada: o relato fica a cargo da bateria da Seção 4.

In [ ]:
def v_disjuncao_por_ensaio(p):
    """Nenhum ensaio pode constar de mais de um conjunto."""
    contagem = p["run_idx"].value_counts()
    repetidos = contagem[contagem > 1].index.tolist()
    assert not repetidos, f"ensaios em mais de um conjunto: {repetidos}"


def v_cobertura(p):
    """Todo ensaio da base pertence a exatamente um conjunto."""
    assert len(p) == len(perfil), f"particao tem {len(p)} linhas, base tem {len(perfil)}"
    esperado = set(range(len(perfil)))
    assert set(p["run_idx"]) == esperado, "ha ensaios fora da particao ou indices inesperados"


def _disjuncao_por_chave(p, k):
    treino = set(k[p.loc[p["conjunto"] == "treino", "run_idx"]])
    teste = set(k[p.loc[p["conjunto"] == "teste", "run_idx"]])
    comuns = treino & teste
    assert not comuns, f"{len(comuns)} configuracao(oes) presente(s) em treino e em teste"


def v_disjuncao_perfil(p):
    """Nenhuma configuracao da versao perfil aparece nos dois conjuntos."""
    _disjuncao_por_chave(p, chave(perfil))


def v_disjuncao_13cat(p):
    """A mesma garantia sob a chave detalhada, que valida a particao para a outra versao."""
    _disjuncao_por_chave(p, chave(cat13))


def v_tipos_no_treino(p):
    """Todo tipo de ensaio esta representado no treino, e no treino de cada dobra."""
    treino = p[p["conjunto"] == "treino"]
    tipos = set(p["run_case"])
    faltando = tipos - set(treino["run_case"])
    assert not faltando, f"tipos ausentes do treino: {sorted(faltando)}"

    tipos_treino = set(treino["run_case"])
    for dobra in sorted(treino["dobra"].dropna().unique()):
        restante = treino[treino["dobra"] != dobra]
        ausentes = tipos_treino - set(restante["run_case"])
        assert not ausentes, f"dobra {int(dobra)} treina sem os tipos {sorted(ausentes)}"


def v_sem_feature_derivada(p, dados=None):
    """Nenhuma coluna de entrada pode ser determinada por uma coluna-alvo."""
    dados = alvos if dados is None else dados
    entrada = colunas_de_configuracao(dados)
    alvo = [c for c in dados.columns if c not in entrada + ["run_idx"]]

    for f in entrada:
        for a in alvo:
            par = pd.concat([dados[f], dados[a]], axis=1).dropna()
            # A correlacao so informa algo quando as duas colunas variam dentro do
            # subconjunto comparado. Colunas constantes produzem correlacao indefinida,
            # e tratar esse caso como violacao geraria alarme falso.
            if len(par) < 3 or par.iloc[:, 0].nunique() < 2 or par.iloc[:, 1].nunique() < 2:
                continue
            r = par.iloc[:, 0].corr(par.iloc[:, 1])
            if pd.isna(r):
                continue
            assert abs(r) < 0.999, (
                f"feature {f} determinada pelo alvo {a} (correlacao {abs(r):.3f})")

## 3. Verificações do tratamento de nulos

As duas verificações seguintes examinam o preenchimento de ausentes. A primeira confere que a estatística vem apenas do treino; a segunda, que nenhum valor preenchido chega ao conjunto avaliado.

As regras são as mesmas do notebook revisado de tratamento de nulos, reimplementadas aqui para que a verificação não dependa da implementação que ela examina.

In [ ]:
PCT_NULOS_MAX, FAIXA_MAX, MULT_EXTREMO = 20.0, 35.0, 2.5


def valor_de_preenchimento(serie):
    """Devolve (acao, valor) segundo as regras revisadas."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    dentro = s[(s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)]
    faixa = dentro.max() - dentro.min() if len(dentro) else 0.0
    extremos = int(((s <= q1 - MULT_EXTREMO * iqr) | (s >= q3 + MULT_EXTREMO * iqr)).sum())
    pct = 100 * int(serie.isna().sum()) / len(serie)

    if pct > PCT_NULOS_MAX and faixa > FAIXA_MAX:
        return "excluir", np.nan
    if extremos > 0:
        return "mediana", s.median()
    return "media", s.mean()


def _mesmo_preenchimento(a, b):
    """Compara pares (acao, valor), tratando ausente como igual a ausente.

    A comparacao direta por np.isclose nao serve: quando a regra decide excluir a
    coluna o valor devolvido e ausente, e ausente nunca e proximo de ausente, de
    modo que a verificacao levantaria alarme falso justamente no caso em que nada
    e preenchido.
    """
    acao_a, valor_a = a
    acao_b, valor_b = b
    if acao_a != acao_b:
        return False
    if valor_a != valor_a and valor_b != valor_b:
        return True
    return bool(np.isclose(valor_a, valor_b))


def v_imputacao_ajustada_no_treino(p, indices_estatistica=None):
    """O valor de preenchimento provem do treino, e nao da base completa."""
    tr = p.loc[p["conjunto"] == "treino", "run_idx"].tolist()
    usados = tr if indices_estatistica is None else indices_estatistica

    usado = valor_de_preenchimento(alvos[ALVO_DEMO].iloc[usados])
    treino = valor_de_preenchimento(alvos[ALVO_DEMO].iloc[tr])
    base = valor_de_preenchimento(alvos[ALVO_DEMO])

    # O alvo de demonstracao precisa ser efetivamente preenchido: se a regra o
    # exclui, nao existe valor a conferir e a verificacao nada afirmaria.
    assert treino[0] != "excluir", (
        f"o alvo {ALVO_DEMO} e excluido pela regra, e nao preenchido; escolha "
        "outro alvo de demonstracao, pois nesta condicao a verificacao e vazia")

    assert _mesmo_preenchimento(usado, treino), (
        f"preenchimento {usado} nao corresponde ao do treino {treino}")
    assert not _mesmo_preenchimento(treino, base), (
        "a estatistica do treino coincide com a da base completa; a verificacao "
        "perde poder de deteccao neste alvo")


def v_avaliacao_sem_imputacao(p, imputar_avaliacao=False):
    """O conjunto avaliado contem apenas valores efetivamente medidos."""
    te = p.loc[p["conjunto"] == "teste", "run_idx"].tolist()
    coluna = alvos[ALVO_DEMO].iloc[te]

    if imputar_avaliacao:
        tr = p.loc[p["conjunto"] == "treino", "run_idx"].tolist()
        _, valor = valor_de_preenchimento(alvos[ALVO_DEMO].iloc[tr])
        # A corrupcao do controle precisa de valor finito. Caso a regra exclua a
        # coluna, o valor vem ausente e o preenchimento nada alteraria, de modo que
        # o controle passaria sem corromper nada e a verificacao pareceria cega.
        if not np.isfinite(valor):
            valor = float(alvos[ALVO_DEMO].iloc[tr].mean())
        assert np.isfinite(valor), (
            f"o alvo {ALVO_DEMO} nao possui valor de preenchimento finito no treino; "
            "o controle nao teria como corromper a avaliacao")
        coluna = coluna.fillna(valor)

    medidos = alvos[ALVO_DEMO].iloc[te].notna()
    assert int(coluna.notna().sum()) == int(medidos.sum()), (
        f"o conjunto avaliado tem {int(coluna.notna().sum())} valores, mas apenas "
        f"{int(medidos.sum())} foram medidos: ha rotulo imputado na avaliacao")

## 4. Casos de controle

Cada função abaixo devolve uma versão corrompida do artefato, violando exatamente uma das propriedades verificadas. São elas que dão sentido à bateria: se a verificação correspondente não falhar sobre a corrupção, ela não protege nada.

In [ ]:
def c_ensaio_duplicado(p):
    """Repete um ensaio de teste no treino."""
    ruim = p.copy()
    linha = ruim[ruim["conjunto"] == "teste"].iloc[0].copy()
    linha["conjunto"] = "treino"
    return pd.concat([ruim, pd.DataFrame([linha])], ignore_index=True)


def c_configuracao_partida(p):
    """Move um unico ensaio de uma configuracao de teste para o treino.

    A configuracao escolhida precisa ter mais de um ensaio: mover o unico ensaio
    de uma configuracao a transferiria inteira para o treino, sem parti-la, e a
    corrupcao pretendida nao ocorreria.
    """
    ruim = p.copy()
    teste = ruim[ruim["conjunto"] == "teste"]
    tamanhos = teste["configuracao"].value_counts()
    multiplas = tamanhos[tamanhos > 1]
    assert len(multiplas), "nenhuma configuracao de teste possui mais de um ensaio"

    cfg = multiplas.index[0]
    alvo_idx = teste[teste["configuracao"] == cfg].index[0]
    ruim.loc[alvo_idx, "conjunto"] = "treino"
    return ruim


def c_particao_pela_chave_fina(p):
    """Divide pela chave de 13 categorias, que e mais fina que a do perfil.

    O resultado respeita a versao detalhada e parte um grupo da versao perfil,
    que e exatamente o motivo pelo qual a chave mais grossa foi escolhida.
    """
    fina, grossa = chave(cat13), chave(perfil)
    mapa = pd.DataFrame({"fina": fina, "grossa": grossa})
    contagem = mapa.groupby("grossa")["fina"].nunique()
    partiveis = contagem[contagem > 1]
    assert len(partiveis), "nenhuma configuracao do perfil se subdivide na versao detalhada"

    alvo_grossa = partiveis.index[0]
    finas = mapa.loc[mapa["grossa"] == alvo_grossa, "fina"].unique()

    ruim = p.copy()
    ruim["conjunto"] = "treino"
    ruim.loc[mapa["fina"] == finas[0], "conjunto"] = "teste"
    ruim.loc[ruim["conjunto"] == "teste", "dobra"] = np.nan
    return ruim


def c_ensaio_ausente(p):
    """Remove um ensaio da particao."""
    return p.iloc[1:].copy()


def c_tipo_fora_do_treino(p):
    """Leva todos os ensaios de um tipo para o teste."""
    ruim = p.copy()
    tipo = ruim["run_case"].value_counts().idxmin()
    ruim.loc[ruim["run_case"] == tipo, "conjunto"] = "teste"
    ruim.loc[ruim["conjunto"] == "teste", "dobra"] = np.nan
    return ruim


def c_feature_derivada(dados):
    """Acrescenta uma coluna de entrada copiada de uma coluna-alvo."""
    ruim = dados.copy()
    entrada = colunas_de_configuracao(ruim)
    alvo = [c for c in ruim.columns if c not in entrada + ["run_idx"]][0]
    ruim["losses_margem_de_seguranca"] = ruim[alvo]
    return ruim

## 5. Execução da bateria

Cada verificação é aplicada aos artefatos reais e à sua versão corrompida. A tabela resultante deve apresentar aprovação na coluna dos dados reais e reprovação na coluna do controle.

In [ ]:
def executa(funcao, *args, **kwargs):
    """Devolve (passou, mensagem)."""
    try:
        funcao(*args, **kwargs)
        return True, ""
    except AssertionError as erro:
        return False, str(erro)


BATERIA = [
    ("Disjuncao por ensaio",
     lambda: executa(v_disjuncao_por_ensaio, particao),
     lambda: executa(v_disjuncao_por_ensaio, c_ensaio_duplicado(particao))),

    ("Cobertura da particao",
     lambda: executa(v_cobertura, particao),
     lambda: executa(v_cobertura, c_ensaio_ausente(particao))),

    ("Disjuncao por configuracao (perfil)",
     lambda: executa(v_disjuncao_perfil, particao),
     lambda: executa(v_disjuncao_perfil, c_configuracao_partida(particao))),

    ("Disjuncao por configuracao (13cat)",
     lambda: executa(v_disjuncao_13cat, particao),
     lambda: executa(v_disjuncao_13cat, c_configuracao_partida(particao))),

    ("Chave grossa protege a versao detalhada",
     lambda: executa(v_disjuncao_perfil, particao),
     lambda: executa(v_disjuncao_perfil, c_particao_pela_chave_fina(particao))),

    ("Tipos de ensaio no treino e nas dobras",
     lambda: executa(v_tipos_no_treino, particao),
     lambda: executa(v_tipos_no_treino, c_tipo_fora_do_treino(particao))),

    ("Ausencia de feature derivada do alvo",
     lambda: executa(v_sem_feature_derivada, particao),
     lambda: executa(v_sem_feature_derivada, particao, c_feature_derivada(alvos))),

    ("Imputacao ajustada somente no treino",
     lambda: executa(v_imputacao_ajustada_no_treino, particao),
     lambda: executa(v_imputacao_ajustada_no_treino, particao,
                     list(range(len(alvos))))),

    ("Avaliacao sem valor imputado",
     lambda: executa(v_avaliacao_sem_imputacao, particao),
     lambda: executa(v_avaliacao_sem_imputacao, particao, True)),
]

resultados = []
for nome, real, controle in BATERIA:
    ok_real, msg_real = real()
    ok_ctrl, msg_ctrl = controle()
    resultados.append({
        "verificacao": nome,
        "dados reais": "passou" if ok_real else "FALHOU",
        "controle": "detectou" if not ok_ctrl else "NAO DETECTOU",
        "mensagem do controle": (msg_ctrl[:70] if not ok_ctrl else ""),
        "mensagem real": (msg_real[:70] if not ok_real else ""),
    })

resultados = pd.DataFrame(resultados)
print(resultados[["verificacao", "dados reais", "controle"]].to_string(index=False))

## 6. Veredito

O notebook falha aqui em duas situações: quando alguma verificação reprova sobre os artefatos reais, indicando vazamento efetivo, e quando alguma verificação passa sobre o controle, indicando que ela não detecta o problema que deveria proteger.

In [ ]:
falhas_reais = resultados[resultados["dados reais"] != "passou"]
controles_cegos = resultados[resultados["controle"] != "detectou"]

if len(falhas_reais):
    print("Verificacoes reprovadas sobre os dados reais:")
    print(falhas_reais[["verificacao", "mensagem real"]].to_string(index=False))
    print()

if len(controles_cegos):
    print("Verificacoes que nao detectaram a corrupcao correspondente:")
    print(controles_cegos[["verificacao"]].to_string(index=False))
    print()

assert len(falhas_reais) == 0, "ha vazamento nos artefatos"
assert len(controles_cegos) == 0, "ha verificacao sem poder de deteccao"

print(f"{len(resultados)} verificacoes aprovadas sobre os artefatos reais.")
print(f"{len(resultados)} casos de controle corretamente detectados.")
print()
print("Nenhum vazamento identificado entre os conjuntos de treino e de teste.")

## 8. Consistência da célula canônica de métricas

As funções de métrica não ficam em módulo, por decisão de manter o projeto inteiramente em notebooks. Elas existem, portanto, em duas cópias: a definição em metricas.ipynb e a cópia em modelagem.ipynb.

Duas cópias divergem em silêncio. Uma correção aplicada a uma e esquecida na outra faria os dois notebooks reportarem métricas diferentes sobre os mesmos dados, sem que nada indicasse o problema. Foi o que ocorreu antes desta verificação existir: a cópia da modelagem havia perdido a linha que soma as contagens de decisão na consolidação por área, e a divergência só apareceu numa comparação manual.

A verificação abaixo compara o conteúdo funcional das duas cópias, desconsiderando linhas em branco e linhas compostas apenas por comentário. Diferenças de comentário são apenas relatadas, uma vez que não alteram resultado; diferenças de código interrompem a execução.

Ela reproduz, para o código, o mesmo princípio das demais verificações deste notebook: transformar um risco silencioso em falha visível.

In [ ]:
import difflib
import json as _json

CAMINHOS = {
    "metricas.ipynb": ["metricas.ipynb",
                       os.path.join("preparacao_testes", "metricas.ipynb"),
                       os.path.join("..", "preparacao_testes", "metricas.ipynb"),
                       os.path.join("notebooks", "preparacao_testes", "metricas.ipynb")],
    "modelagem.ipynb": ["modelagem.ipynb",
                        os.path.join("colabs_S3", "modelagem.ipynb"),
                        os.path.join("..", "modelagem.ipynb"),
                        os.path.join("notebooks", "colabs_S3", "modelagem.ipynb")],
}


def _canonica(nome):
    """Extrai a celula que define as metricas, ou None se o arquivo nao existir."""
    for caminho in CAMINHOS[nome]:
        if not os.path.exists(caminho):
            continue
        nb = _json.load(open(caminho, encoding="utf-8"))
        for c in nb["cells"]:
            s = "".join(c["source"])
            if c["cell_type"] == "code" and "def avaliar(" in s and "def consolidar(" in s:
                return s
    return None


def _codigo(fonte):
    """Linhas com efeito, sem branco e sem linha exclusivamente de comentario."""
    linhas = []
    for linha in fonte[fonte.index("MARGEM_KPI"):].split("\n"):
        limpa = linha.rstrip()
        if not limpa.strip() or limpa.strip().startswith("#"):
            continue
        linhas.append(limpa)
    return linhas


fontes = {nome: _canonica(nome) for nome in CAMINHOS}
ausentes = [n for n, f in fontes.items() if f is None]

if ausentes:
    print("Verificacao nao realizada: nao encontrados ->", ", ".join(ausentes))
    print("No Colab, suba os dois notebooks ao lado deste arquivo para habilita-la.")
else:
    a, b = _codigo(fontes["metricas.ipynb"]), _codigo(fontes["modelagem.ipynb"])
    divergencias = [l for l in difflib.unified_diff(a, b, lineterm="", n=0)
                    if l[:1] in "+-" and l[:3] not in ("+++", "---")]

    print(f"metricas.ipynb : {len(a)} linhas de codigo")
    print(f"modelagem.ipynb: {len(b)} linhas de codigo")
    print()
    if divergencias:
        print("DIVERGENCIA entre as copias da celula canonica:")
        for linha in divergencias[:20]:
            print("  ", linha)
    else:
        print("As duas copias sao identicas no conteudo funcional.")

    assert not divergencias, (
        "as copias da celula canonica de metricas divergem; "
        "sincronize a partir de metricas.ipynb, que e a fonte")

## 9. O que estas verificações não cobrem

Três limites devem ser registrados, para que a aprovação não seja lida como garantia mais ampla do que é.

O escopo é o dos artefatos, e não o do código que os produz. As verificações examinam a partição gravada e os conjuntos por área; um defeito no notebook de preparação que produzisse arquivos internamente consistentes, porém errados, não seria detectado aqui.

A verificação de feature derivada apoia-se em correlação linear, e detecta cópia direta ou transformação afim de uma coluna-alvo. Uma dependência não linear entre entrada e alvo passaria despercebida.

A verificação da imputação examina uma coluna representativa, escolhida por ser a de menor cobertura da base. A extensão a todas as colunas com ausências é possível e recomendável quando o conjunto de alvos estiver estabilizado.